# Phase 4 (v4) — Ablation study (Models A–E) on the cleaned data (features_v3)
* Data: features_v3 (invalid values, duplicate feature vectors and conflicting labels removed; Section 3.6.4)
* 3,000,000-row stratified subset of each training split (all rows when the split is smaller) — Wahab et al. (2025)
* 3-fold stratified cross-validation — Wahab et al. (2025)
* Class imbalance: class weights w_c = N / (K * N_c), computed on each training fold only — Kamal & Mashaly (2025)
* Adam, learning rate 1e-4, cross-entropy, batch size 128, up to 100 epochs, dropout 0.3 — Wahab et al. (2025)
* Early stopping: patience 5 on the loss of the complete validation split, best weights restored — Wahab et al. (2025), Sec. 4.8.6
* Fault tolerance: after every epoch the full model (weights + optimizer), the best weights and the early-stopping
  state are written to Drive with atomic replacement; an interrupted run resumes from its last finished epoch.
* Score: Macro F1 on the held-out fold — C. Wang et al. (2024)
* Selection rule (fixed before the run): mean Macro F1 over folds and configurations (Wahab et al., 2025);
  corrected resampled t-test (Nadeau & Bengio, 2003); one-standard-error rule (Hastie et al., 2009).
Outputs: outputs_v3/ablation_v4/ (the earlier ablation in outputs_v3/ablation/ is not touched).

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings, models, helpers
import os, json, time, gc, math, platform
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

FEAT    = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
OUT     = '/content/drive/MyDrive/Thesis/outputs_v3/ablation_v4/'
os.makedirs(OUT + 'results/', exist_ok=True); os.makedirs(OUT + 'checkpoints/', exist_ok=True)
CONFIGS = ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']
MODELS  = ['A', 'B', 'C', 'D', 'E']
SEED, SUBSET, FOLDS = 42, 3_000_000, 3
LR, BATCH, EPOCHS, PATIENCE = 1e-4, 128, 100, 5
N_CLASSES = 8
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
TOTAL_RUNS = len(CONFIGS) * FOLDS * len(MODELS)
LOGFILE = OUT + 'ablation_v4_log.txt'

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:                        # the full log is also kept on Drive
        f.write(line + '\n')

def hm(minutes):
    minutes = max(0.0, float(minutes))
    return f'{int(minutes // 60)} h {int(round(minutes % 60)):02d} min' if minutes >= 60 else f'{minutes:.1f} min'

def eca(x):
    c = x.shape[-1]
    w = layers.GlobalAveragePooling1D()(x)
    w = layers.Reshape((c, 1))(w)
    w = layers.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w)
    w = layers.Reshape((1, c))(w)
    return layers.Multiply()([x, w])

def build(variant, n_features, n_classes=N_CLASSES):
    """A: CNN+GAP | B: +ECA | C: +skip | D: +ECA +skip | E: +2xECA +skip (identical to the first ablation)"""
    inp = keras.Input((n_features, 1))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    if variant == 'E':
        x = eca(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if variant in ('B', 'D', 'E'):
        x = eca(x)
    z = layers.GlobalAveragePooling1D()(x)
    if variant in ('C', 'D', 'E'):
        s = layers.Dense(128, activation='relu')(layers.Flatten()(inp))
        z = layers.Concatenate()([z, s])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax')(z)
    return keras.Model(inp, out, name=f'Model_{variant}')

def counts(y):
    c = np.bincount(y, minlength=N_CLASSES)
    return {CLASS_NAMES[i]: int(n) for i, n in enumerate(c) if n}

def class_weights(y):
    """w_c = N / (K * N_c) over the K classes present in the training fold (Kamal & Mashaly, 2025)."""
    c = np.bincount(y, minlength=N_CLASSES); present = np.flatnonzero(c)
    return {int(k): float(len(y) / (len(present) * c[k])) for k in present}

def result_path(cfg, fold, m):
    return OUT + f'results/{cfg}_fold{fold}_model{m}.json'

def ckpt_dir(cfg, fold, m):
    return OUT + f'checkpoints/{cfg}_fold{fold}_model{m}/'

def n_done():
    return sum(os.path.exists(result_path(c, f, m)) for c in CONFIGS for f in range(FOLDS) for m in MODELS)

def avg_run_minutes():
    r = [json.load(open(OUT + 'results/' + f))['train_minutes'] for f in os.listdir(OUT + 'results/') if f.endswith('.json')]
    return (float(np.mean(r)) if r else None), len(r)

def save_atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def val_metrics(model, Xv, yv):
    """Loss, accuracy, Macro F1 and per-class F1 on the complete validation split (one prediction pass)."""
    p = model.predict(Xv, batch_size=8192, verbose=0)
    loss = float(-np.mean(np.log(np.clip(p[np.arange(len(yv)), yv], 1e-7, 1.0))))   # unweighted cross-entropy
    yp = p.argmax(1); labels = np.unique(yv)
    f1c = f1_score(yv, yp, average=None, labels=labels, zero_division=0)
    return loss, float(accuracy_score(yv, yp)), float(f1c.mean()), {CLASS_NAMES[k]: round(float(v), 3) for k, v in zip(labels, f1c)}

class ProgressCheckpoint(keras.callbacks.Callback):
    """Detailed progress (every 10% of an epoch, with time left), complete-validation evaluation, early stopping
    (patience 5, best weights kept) and an atomic checkpoint after every epoch."""
    def __init__(self, tag, d, Xv, yv, steps, state):
        super().__init__()
        self.tag, self.d, self.Xv, self.yv, self.steps, self.state = tag, d, Xv, yv, steps, state
        self.marks = {max(1, int(steps * q / 10)) for q in range(1, 10)}
        self.t_run = time.time() - state['minutes'] * 60
    def on_epoch_begin(self, epoch, logs=None):
        self.t_ep, self.cur = time.time(), epoch + 1
        log(f"{self.tag} | epoch {self.cur} started ({self.steps:,} steps of {BATCH})")
    def on_train_batch_end(self, batch, logs=None):
        if batch + 1 in self.marks:
            el = (time.time() - self.t_ep) / 60; frac = (batch + 1) / self.steps
            log(f"{self.tag} | epoch {self.cur} | step {batch + 1:,}/{self.steps:,} ({100 * frac:.0f}%) | "
                f"running loss {logs['loss']:.4f} | {hm(el)} elapsed, about {hm(el / frac - el)} left in this epoch")
    def on_epoch_end(self, epoch, logs=None):
        st, d = self.state, self.d
        t_train = (time.time() - self.t_ep) / 60
        vloss, vacc, vf1, vpc = val_metrics(self.model, self.Xv, self.yv)
        improved = vloss < st['best']
        if improved:
            self.model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
            st.update(best=vloss, best_ep=epoch + 1, wait=0)
        else:
            st['wait'] += 1
        st.update(epoch=epoch + 1, minutes=(time.time() - self.t_run) / 60)
        st.setdefault('history', []).append(dict(epoch=epoch + 1, loss=round(float(logs['loss']), 5), val_loss=round(vloss, 5),
                                                 val_acc=round(vacc, 5), val_macro_f1=round(vf1, 5)))
        self.model.save(d + 'last.tmp.keras'); os.replace(d + 'last.tmp.keras', d + 'last.keras')
        save_atomic_json(st, d + 'state.json')
        ep_min = st['minutes'] / st['epoch']
        log(f"{self.tag} | epoch {epoch + 1} done | train {t_train:.1f} min + validation {(time.time() - self.t_ep) / 60 - t_train:.1f} min | "
            f"loss {logs['loss']:.4f} | val_loss {vloss:.4f}{' (best)' if improved else ''} | val_acc {vacc:.4f} | val_macroF1 {vf1:.4f}")
        log(f"{self.tag} |   val F1 per class: {vpc}")
        log(f"{self.tag} |   best epoch {st['best_ep']} | no improvement {st['wait']}/{PATIENCE} | run time {hm(st['minutes'])} | "
            f"at most {PATIENCE - st['wait']} more epochs without improvement (about {hm((PATIENCE - st['wait']) * ep_min)}) | checkpoint saved")
        if st['wait'] >= PATIENCE:
            log(f"{self.tag} | early stopping: no improvement for {PATIENCE} epochs; best epoch {st['best_ep']} will be restored")
            self.model.stop_training = True

log('=' * 100)
log(f'Ablation v4 | TensorFlow {tf.__version__} | Python {platform.python_version()} | GPU: '
    f'{[g.name for g in tf.config.list_physical_devices("GPU")] or "none"}')
log(f'settings: subset {SUBSET:,}, {FOLDS} folds, lr {LR}, batch {BATCH}, max epochs {EPOCHS}, patience {PATIENCE}, '
    f'class weights N/(K*N_c), complete validation split | {n_done()} of {TOTAL_RUNS} runs already finished')

In [ ]:
# %% Cell 2 — run the ablation (resumable: re-run this cell after any interruption)
T0 = time.time()
for cfg in CONFIGS:
    if all(os.path.exists(result_path(cfg, f, m)) for f in range(FOLDS) for m in MODELS):
        log(f'{cfg}: all 15 runs done, skipping'); continue
    t_load = time.time()
    X = np.load(FEAT + cfg + '/X_train.npy', mmap_mode='r'); y = np.load(FEAT + cfg + '/y_train.npy')
    if len(y) > SUBSET:
        idx, _ = train_test_split(np.arange(len(y)), train_size=SUBSET, stratify=y, random_state=SEED); idx = np.sort(idx)
    else:
        idx = np.arange(len(y))
    Xs, ys = np.asarray(X[idx], dtype=np.float32), y[idx].astype(np.int64)
    Xv = np.asarray(np.load(FEAT + cfg + '/X_val.npy'), dtype=np.float32)[..., None]
    yv = np.load(FEAT + cfg + '/y_val.npy').astype(np.int64)
    assert np.isfinite(Xs).all() and np.isfinite(Xv).all(), 'non-finite values in the data'
    log(f'{cfg}: training split {len(y):,} rows -> subset {len(ys):,} rows, {Xs.shape[1]} features | complete validation split '
        f'{len(yv):,} rows | value range [{Xs.min():.3f}, {Xs.max():.3f}] | loaded in {time.time() - t_load:.0f}s')
    log(f'{cfg}: subset class counts {counts(ys)}')
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
    for fold, (tr, te) in enumerate(skf.split(Xs, ys)):
        todo = [m for m in MODELS if not os.path.exists(result_path(cfg, fold, m))]
        if not todo:
            continue
        Xtr, ytr, Xte, yte = Xs[tr][..., None], ys[tr], Xs[te][..., None], ys[te]
        cw = class_weights(ytr); steps = int(math.ceil(len(ytr) / BATCH))
        log(f'{cfg} fold {fold}: training fold {len(ytr):,} rows, held-out fold {len(yte):,} rows')
        log(f'{cfg} fold {fold}: training class counts {counts(ytr)}')
        log(f'{cfg} fold {fold}: class weights ' + str({CLASS_NAMES[k]: round(v, 4) for k, v in cw.items()}))
        for m in todo:
            tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
            d = ckpt_dir(cfg, fold, m); os.makedirs(d, exist_ok=True)
            tag = f'{cfg} f{fold} {m}'
            head = f'--- run {n_done() + 1} of {TOTAL_RUNS}: {cfg}, fold {fold}, Model {m}'
            if os.path.exists(d + 'state.json') and os.path.exists(d + 'last.keras'):
                state = json.load(open(d + 'state.json'))
                model = keras.models.load_model(d + 'last.keras')
                ep_model = int(keras.ops.convert_to_numpy(model.optimizer.iterations)) // steps
                log(f"{head} --- RESUMING after epoch {state['epoch']} (best epoch {state['best_ep']}, "
                    f"no improvement {state['wait']}/{PATIENCE}, {hm(state['minutes'])} already trained) ---")
                if ep_model == state['epoch'] + 1:        # interrupted between saving the model and saving the state
                    log(f'{tag} | the saved model is one epoch ahead of the saved state; evaluating it to bring the state up to date')
                    vloss = val_metrics(model, Xv, yv)[0]
                    if vloss < state['best']:
                        model.save_weights(d + 'best.tmp.weights.h5'); os.replace(d + 'best.tmp.weights.h5', d + 'best.weights.h5')
                        state.update(best=vloss, best_ep=ep_model, wait=0)
                    else:
                        state['wait'] += 1
                    state['epoch'] = ep_model; save_atomic_json(state, d + 'state.json')
                elif ep_model != state['epoch']:
                    log(f"{tag} | note: optimizer step count corresponds to epoch {ep_model}, state says {state['epoch']}; continuing from the state")
            else:
                state = dict(epoch=0, best=float('inf'), best_ep=0, wait=0, minutes=0.0, history=[])
                model = build(m, Xs.shape[1])
                model.compile(optimizer=keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy')
                log(f'{head} ---')
            params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
            avg, nres = avg_run_minutes()
            log(f'{tag} | {params:,} trainable parameters | {steps:,} steps per epoch | '
                + (f'average finished run {hm(avg)}, so about {hm(avg * (TOTAL_RUNS - n_done()))} left for the whole ablation'
                   if avg else 'time estimates start after the first finished run'))
            if not (state['wait'] >= PATIENCE or state['epoch'] >= EPOCHS):
                model.fit(Xtr, ytr, batch_size=BATCH, epochs=EPOCHS, initial_epoch=state['epoch'], verbose=0, shuffle=True,
                          class_weight=cw, callbacks=[ProgressCheckpoint(tag, d, Xv, yv, steps, state)])
            model.load_weights(d + 'best.weights.h5')                  # best epoch (early stopping)
            t_ev = time.time()
            pred = model.predict(Xte, batch_size=8192, verbose=0).argmax(1)
            labels = np.unique(yte)
            pr, rc, f1c, sup = precision_recall_fscore_support(yte, pred, labels=labels, zero_division=0)
            res = dict(config=cfg, fold=fold, model=m, epochs=state['epoch'], best_epoch=state['best_ep'],
                       train_minutes=round(state['minutes'], 2), params=params,
                       macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(yte, pred)),
                       weighted_f1=float(f1_score(yte, pred, average='weighted', labels=labels, zero_division=0)),
                       per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), support=int(s_))
                                  for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)},
                       confusion_matrix=confusion_matrix(yte, pred, labels=labels).tolist(),
                       labels=[CLASS_NAMES[k] for k in labels], class_weights={CLASS_NAMES[k]: v for k, v in cw.items()},
                       train_rows=int(len(ytr)), test_rows=int(len(yte)), history=state.get('history', []))
            save_atomic_json(res, result_path(cfg, fold, m))
            log(f"{cfg} fold {fold} Model {m}: RESULT Macro F1 {res['macro_f1']:.4f} | accuracy {res['accuracy']:.4f} | "
                f"weighted F1 {res['weighted_f1']:.4f} | {res['epochs']} epochs (best {res['best_epoch']}) | {hm(res['train_minutes'])} | "
                f"evaluated in {time.time() - t_ev:.0f}s")
            log('   per class (precision / recall / F1): ' + '; '.join(
                f"{k} {v['precision']:.3f}/{v['recall']:.3f}/{v['f1']:.3f}" for k, v in res['per_class'].items()))
            avg, nres = avg_run_minutes()
            log(f'   finished {nres} of {TOTAL_RUNS} runs | average {hm(avg)} per run | about {hm(avg * (TOTAL_RUNS - nres))} left | '
                f'this session running for {hm((time.time() - T0) / 60)}')
            del model; gc.collect()
        del Xtr, ytr, Xte, yte; gc.collect()
    del Xs, ys, Xv, yv; gc.collect()
log(f'ablation v4: {n_done()} of {TOTAL_RUNS} runs finished')

In [ ]:
# %% Cell 3 — results and the selection rule fixed before the run
from scipy import stats
rows = [json.load(open(OUT + 'results/' + f)) for f in sorted(os.listdir(OUT + 'results/')) if f.endswith('.json')]
df = pd.DataFrame(rows)[['config', 'fold', 'model', 'macro_f1', 'accuracy', 'epochs', 'best_epoch', 'train_minutes', 'params']]
print(f'{len(df)} of {TOTAL_RUNS} runs finished\n')
if len(df) == TOTAL_RUNS:
    print('Macro F1 per fold:'); print(df.pivot_table(index=['config', 'model'], columns='fold', values='macro_f1').round(4).to_string())
    g = df.groupby(['config', 'model'])['macro_f1']
    print('\nMean ± std over the 3 folds:'); print(pd.DataFrame({'mean': g.mean(), 'std': g.std()}).round(4).to_string())
    piv = df.groupby(['model', 'config'])['macro_f1'].mean().unstack(); piv['average'] = piv.mean(axis=1)
    print('\nStep 1-2 (Wahab et al., 2025): mean Macro F1 per configuration and average over configurations:')
    print(piv.round(4).sort_values('average', ascending=False).to_string())
    best = piv['average'].idxmax()
    def nb(a, b, k=FOLDS, ratio=1 / (FOLDS - 1)):        # corrected resampled t-test (Nadeau & Bengio, 2003)
        dlt = np.asarray(a) - np.asarray(b); v = dlt.var(ddof=1)
        t = dlt.mean() / math.sqrt((1 / k + ratio) * v) if v > 0 else float('inf')
        return dlt.mean(), 2 * stats.t.sf(abs(t), k - 1)
    print(f'\nStep 3 (Nadeau & Bengio, 2003): best variant {best} against each other variant, per configuration')
    for c in CONFIGS:
        s = df[df.config == c]
        for m in MODELS:
            if m != best:
                dm, p = nb(s[s.model == best].sort_values('fold').macro_f1, s[s.model == m].sort_values('fold').macro_f1)
                print(f'   {c:20s} {best} - {m}: difference {dm:+.4f}, p = {p:.3f}')
    fold_avg = df.groupby(['model', 'fold'])['macro_f1'].mean().unstack()
    se = fold_avg.loc[best].std(ddof=1) / math.sqrt(FOLDS); thr = piv.loc[best, 'average'] - se
    within = [m for m in MODELS if piv.loc[m, 'average'] >= thr]
    mean_params = df.groupby('model')['params'].mean()
    chosen = min(within, key=lambda m: (mean_params[m], -piv.loc[m, 'average']))
    print(f'\nStep 4 (Hastie et al., 2009): best mean {piv.loc[best, "average"]:.4f}, standard error {se:.4f}, threshold {thr:.4f}')
    print(f'   variants within one standard error: {within} | mean parameters: {mean_params[within].astype(int).to_dict()}')
    print(f'\nSELECTED ARCHITECTURE: Model {chosen}')
    save_atomic_json(dict(best_mean=best, selected=chosen, threshold=float(thr), se=float(se), within_one_se=within,
                          averages=piv['average'].round(6).to_dict()), OUT + 'selection.json')